In [1]:
import os, glob
import sounddevice as sd
from scipy.io.wavfile import write
from src.constants import Constants as C
from src.levenshtein import damerau_levenshtein_weighted
from src.levenshtein import damerau_levenshtein_neighbour_aware
from src.parsers import wav_to_logmel
from src.evaluator import evaluate_word
import numpy as np
import torch
import torch.nn as nn
import torchaudio.transforms as T
from src.constants import Constants as C
from pathlib import Path

from src.parsers import PhonemeWindowDataset
from src.NeuralModel import CRNN
from src.trainers import  load_checkpoint
from src.evaluator import evaluate_audio
from src.wordmaker import PHONEME_TO_LETTERS, levenshtein_distance, phonemes_to_text, parse_words, WLIST1000, proba_predict
 


CHECKPOINT_PATH = "../trained_models/BetterDataSoft.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CRNN()
meta = load_checkpoint(CHECKPOINT_PATH, model, device=device)
model.eval()

print("checkpoint meta keys:", list(meta.keys()))

checkpoint meta keys: []


In [ ]:
import gradio as gr

def transcribe(audio_path):
    result = evaluate_audio(audio_path, model=model, device=device,
                             show_per_window=False, top_k=4)
    phonemes = proba_predict(result, p=0.67, longer_reg=True, aeo_reg=True)
    text = phonemes_to_text(phonemes, after_silence=False)
    
    output = min(WLIST1000, key=lambda w: damerau_levenshtein_weighted(w, phonemes))
    min_dist = damerau_levenshtein_weighted(output, phonemes)
    
    return ' '.join(phonemes), text, output.upper(), f'{min_dist:.2f}'


demo = gr.Interface(
    fn=transcribe,
    inputs=gr.Audio(sources=['microphone'], type='filepath'),
    outputs=[
        gr.Textbox(label='Fonemy'),
        gr.Textbox(label='Transkrypcja'),
        gr.Textbox(label='Najlepsze dopasowanie'),
        gr.Textbox(label='Distance'),
    ],
    title='🎙️ Phoneme Recognizer',
    theme=gr.themes.Soft(),
)

demo.launch()

/home/stachuapa123/Desktop/ASR/ASR_project/.venv/lib/python3.13/site-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/stachuapa123/Desktop/ASR/ASR_project/.venv/lib/python3.13/site-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/home/stachuapa123/Desktop/ASR/ASR_project/.venv/lib/python3.13/site-packages/gradio/route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/home/stachuapa123/Desktop/ASR/ASR_project/.venv/lib/python3.13/site-packages/gradio/blocks.py", line 2195, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "/home/stachuapa123/Desktop/ASR/ASR_project/.venv/lib/python3.13/site-packages/gradio/blocks.py", line 1652, in call_function
    prediction = await anyio.to_thread.run_sync(  #